# Merge Portions of NBA Data Together

In [1]:
# ▸ Cell 1 ─────────────────────────────────────────────────────────────
# merge_data.ipynb  – v3
# Build a two-row-per-game DataFrame from Kaggle's game.csv + line_score.csv
#   • renames *_home / *_away → *_main / *_opp  in BOTH files (pre-merge)
#   • merges on game_id
#   • duplicates each game with roles flipped (iscopy = 0 / 1)
# ---------------------------------------------------------------------
import pandas as pd
from pathlib import Path

# Notebook lives in ./notebooks/
DATA_DIR       = Path("../data")
RAW_DIR        = DATA_DIR / "raw"
CSV_DIR        = RAW_DIR  / "csv"
PROCESSED_DIR  = DATA_DIR / "processed"
PROCESSED_DIR.mkdir(exist_ok=True)

print("✓ Directory layout checked")

✓ Directory layout checked


In [2]:
# ▸ Cell 2 ─────────────────────────────────────────────────────────────
# 1. Load *all* columns
# ---------------------------------------------------------------------
game_df = pd.read_csv(CSV_DIR / "game.csv")          # every column
line_df = pd.read_csv(CSV_DIR / "line_score.csv")    # every column

print(f"game_df shape : {game_df.shape}")
print(f"line_df shape : {line_df.shape}")

game_df shape : (65698, 55)
line_df shape : (58053, 43)


In [3]:
# ▸ Cell 3 ─────────────────────────────────────────────────────────────
# 2. Rename *_home / *_away columns → *_main / *_opp    (in BOTH DataFrames)
# ---------------------------------------------------------------------
def rename_home_away(df: pd.DataFrame) -> pd.DataFrame:
    """Return a copy with *_home / *_away renamed to *_main / *_opp."""
    home_cols = [c for c in df.columns if c.endswith("_home")]
    away_cols = [c for c in df.columns if c.endswith("_away")]
    rename_map = {c: f"{c[:-5]}_main" for c in home_cols} | \
                 {c: f"{c[:-5]}_opp"  for c in away_cols}
    return df.rename(columns=rename_map)

line_df = rename_home_away(line_df)
game_df = rename_home_away(game_df)

print("✓ Renamed suffixes in both tables")

✓ Renamed suffixes in both tables


In [4]:
# ▸ Cell 4 ─────────────────────────────────────────────────────────────
# 3. Merge on game_id  (line_df = “left”, game_df fields get *_game suffix)
# ---------------------------------------------------------------------
wide = (
    line_df
      .merge(game_df, on="game_id", how="left", suffixes=("", "_game"))
)

missing = wide["game_date"].isna().sum()
print(f"merge → {wide.shape} rows   |   unmatched game_ids = {missing}")

merge → (58149, 97) rows   |   unmatched game_ids = 0


In [5]:
# ▸ Cell 5 ─────────────────────────────────────────────────────────────
# 4. Build two orientations
#    • iscopy = 0 : home team is MAIN  (already true)
#    • iscopy = 1 : away team is MAIN  (swap values in *_main / *_opp pairs)
# ---------------------------------------------------------------------
home_main = wide.assign(iscopy=0)          # orientation 1

away_main = wide.copy()                    # orientation 2 (to be flipped)
# Identify every column whose *name* marks it as "main"
main_cols = [c for c in wide.columns if "_main" in c]

# Swap each *_main ↔ *_opp partner (handles *_main_game too)
for main_col in main_cols:
    opp_col = main_col.replace("_main", "_opp")
    if opp_col in away_main.columns:
        away_main[[main_col, opp_col]] = away_main[[opp_col, main_col]].to_numpy()

away_main = away_main.assign(iscopy=1)

# Stack the two tables
full_nba = pd.concat([home_main, away_main], ignore_index=True)

print(f"✓ full_nba shape : {full_nba.shape}  (expected 2 × line_df rows)")

✓ full_nba shape : (116298, 98)  (expected 2 × line_df rows)


In [6]:
# ▸ Cell 6 ─────────────────────────────────────────────────────────────
# 5. Quick peek
# ---------------------------------------------------------------------
show_cols = [c for c in full_nba.columns if c.endswith(("_main", "_opp"))][:10]
print("Sample paired columns:", show_cols)
full_nba.head()

Sample paired columns: ['team_id_main', 'team_abbreviation_main', 'team_city_name_main', 'team_nickname_main', 'team_wins_losses_main', 'pts_qtr1_main', 'pts_qtr2_main', 'pts_qtr3_main', 'pts_qtr4_main', 'pts_ot1_main']


,game_date_est,game_sequence,game_id,team_id_main,team_abbreviation_main,team_city_name_main,team_nickname_main,team_wins_losses_main,pts_qtr1_main,pts_qtr2_main,...,ast_opp,stl_opp,blk_opp,tov_opp,pf_opp,pts_opp_game,plus_minus_opp,video_available_opp,season_type,iscopy
0,1946-11-01 00:00:00,NaN,24600001,1610610035,HUS,Toronto,Huskies,-,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,68.0,2,0,Regular Season,0
1,1946-11-02 00:00:00,NaN,24600003,1610610034,BOM,St. Louis,Bombers,-,16.0,16.0,...,NaN,NaN,NaN,NaN,25.0,51.0,-5,0,Regular Season,0
2,1946-11-02 00:00:00,NaN,24600002,1610612738,BOS,Boston,Celtics,-,10.0,16.0,...,NaN,NaN,NaN,NaN,NaN,53.0,-6,0,Regular Season,0
3,1946-11-02 00:00:00,NaN,24600004,1610610025,CHS,Chicago,Stags,-,NaN,NaN,...,NaN,NaN,NaN,NaN,22.0,47.0,-16,0,Regular Season,0
4,1946-11-02 00:00:00,NaN,24600005,1610610036,WAS,Washington,Capitols,-,21.0,4.0,...,NaN,NaN,NaN,NaN,NaN,50.0,17,0,Regular Season,0


In [7]:
# ▸ Cell 7 ─────────────────────────────────────────────────────────────
# Remove any *_game columns (they’re duplicates from game.csv)
full_nba = full_nba.loc[:, ~full_nba.columns.str.endswith("_game")]
# -----------------------------------------------------------------------

In [8]:
# ▸ Cell 8 ──────────────────────────────────────────────────────────
# Normalise the date field: keep only calendar date, drop old columns
# --------------------------------------------------------------------
# 1. Build the new date_game column from game_date_est
full_nba["date_game"] = pd.to_datetime(full_nba["game_date_est"]).dt.date

# 2. Drop the superseded columns
full_nba = full_nba.drop(columns=["game_date_est", "game_date"], errors="ignore")


In [9]:
# ▸ Cell 9 ──────────────────────────────────────────────────────────
# Re-order columns:
#   1) game_id
#   2) date_game
#   3) *_main  columns (in the order they already appear)
#   4) *_opp   columns
#   5) everything else (e.g. iscopy)
# --------------------------------------------------------------------
main_cols  = [c for c in full_nba.columns if c.endswith("_main")]
opp_cols   = [c for c in full_nba.columns if c.endswith("_opp")]

# Preserve original order for any remaining columns
other_cols = [
    c for c in full_nba.columns
    if c not in {"game_id", "date_game"} | set(main_cols) | set(opp_cols)
]

new_order = ["game_id", "date_game"] + main_cols + opp_cols + other_cols
full_nba = full_nba[new_order]

print("✓ Columns reordered")


✓ Columns reordered


In [10]:
# ▸ Cell 10 ──────────────────────────────────────────────────────────
full_nba.head()

,game_id,date_game,team_id_main,team_abbreviation_main,team_city_name_main,team_nickname_main,team_wins_losses_main,pts_qtr1_main,pts_qtr2_main,pts_qtr3_main,...,blk_opp,tov_opp,pf_opp,plus_minus_opp,video_available_opp,game_sequence,season_id,min,season_type,iscopy
0,24600001,1946-11-01,1610610035,HUS,Toronto,Huskies,-,NaN,NaN,NaN,...,NaN,NaN,NaN,2,0,NaN,21946,0,Regular Season,0
1,24600003,1946-11-02,1610610034,BOM,St. Louis,Bombers,-,16.0,16.0,18.0,...,NaN,NaN,25.0,-5,0,NaN,21946,0,Regular Season,0
2,24600002,1946-11-02,1610612738,BOS,Boston,Celtics,-,10.0,16.0,14.0,...,NaN,NaN,NaN,-6,0,NaN,21946,0,Regular Season,0
3,24600004,1946-11-02,1610610025,CHS,Chicago,Stags,-,NaN,NaN,NaN,...,NaN,NaN,22.0,-16,0,NaN,21946,0,Regular Season,0
4,24600005,1946-11-02,1610610036,WAS,Washington,Capitols,-,21.0,4.0,12.0,...,NaN,NaN,NaN,17,0,NaN,21946,0,Regular Season,0


In [11]:
# ▸ Cell 11 ──────────────────────────────────────────────────────────
full_nba.columns

Index(['game_id', 'date_game', 'team_id_main', 'team_abbreviation_main',
       'team_city_name_main', 'team_nickname_main', 'team_wins_losses_main',
       'pts_qtr1_main', 'pts_qtr2_main', 'pts_qtr3_main', 'pts_qtr4_main',
       'pts_ot1_main', 'pts_ot2_main', 'pts_ot3_main', 'pts_ot4_main',
       'pts_ot5_main', 'pts_ot6_main', 'pts_ot7_main', 'pts_ot8_main',
       'pts_ot9_main', 'pts_ot10_main', 'pts_main', 'team_name_main',
       'matchup_main', 'wl_main', 'fgm_main', 'fga_main', 'fg_pct_main',
       'fg3m_main', 'fg3a_main', 'fg3_pct_main', 'ftm_main', 'fta_main',
       'ft_pct_main', 'oreb_main', 'dreb_main', 'reb_main', 'ast_main',
       'stl_main', 'blk_main', 'tov_main', 'pf_main', 'plus_minus_main',
       'video_available_main', 'team_id_opp', 'team_abbreviation_opp',
       'team_city_name_opp', 'team_nickname_opp', 'team_wins_losses_opp',
       'pts_qtr1_opp', 'pts_qtr2_opp', 'pts_qtr3_opp', 'pts_qtr4_opp',
       'pts_ot1_opp', 'pts_ot2_opp', 'pts_ot3_opp', 'pts

In [14]:
# ▸ Cell 12 ─────────────────────────────────────────────────────────────
# Save the tidy full_nba table   (choose the format you prefer)
# ---------------------------------------------------------------------
out_path = PROCESSED_DIR / "full_nba.csv"
full_nba.to_csv(out_path, index=False)

print(f"✓ Saved → {out_path}") 


✓ Saved → ..\data\processed\full_nba.csv
